理解幻觉的本质

In [6]:
from zai import ZhipuAiClient
from dotenv import load_dotenv
from pydantic import BaseModel
from enum import Enum
import asyncio
import pandas as pd

pd.set_option('display.max_rows', None)      # 显示所有行
pd.set_option('display.max_columns', None)   # 显示所有列
pd.set_option('display.max_colwidth', None)  # 单元格内容不截断
pd.set_option('display.width', None)         # 自动适应终端宽度

load_dotenv()
client = ZhipuAiClient()

class ROLE(str, Enum):
    user = "user"
    assistant = "assistant"

class Usage(BaseModel):
    prompt_tokens: int = 0
    completion_tokens: int = 0
    cache_tokens: int = 0
    total_tokens: int = 0
    @property
    def output(self) -> str:
        return f"提示词:{self.prompt_tokens} 回复内容:{self.completion_tokens} 缓存:{self.cache_tokens} 共计:{self.total_tokens}"

class ContentModel(BaseModel):
    content: str
    usage: Usage

class MessageModel(BaseModel):
    role: ROLE
    content: str

def fetch(kwargs:dict) -> ContentModel:
    result = client.chat.completions.create(**kwargs)
    content = result.choices[0].message.content
    prompt_tokens = result.usage.prompt_tokens
    completion_tokens = result.usage.completion_tokens
    cache_tokens = result.usage.prompt_tokens_details.cached_tokens
    total_tokens = result.usage.total_tokens
    usage = Usage(
        prompt_tokens=prompt_tokens,
        completion_tokens=completion_tokens,
        cache_tokens=cache_tokens,
        total_tokens=total_tokens
    )
    return ContentModel(content=content, usage=usage)

In [7]:
def chat(
    input: str, 
    temperature: float | None = None, 
    do_sample: bool | None = None
) -> ContentModel:
    paras = {
        "model": "glm-4.5-air",
        "messages": [
            {"role": ROLE.user, "content": input},
        ],
        "stream": False,
    }
    if temperature is not None:
        paras["temperature"] = temperature
    if do_sample is not None:
        paras["do_sample"] = do_sample

    result = fetch(paras)

    return result

In [9]:
    prompts = [
        #"请介绍一下作家张伟立 2023 年出版的小说《雾港深处》的剧情",
        "Swift 6 里新增的 @AsyncReentrant 属性包装器怎么用?",
        # "神州23号飞船发射的时候，当时央视直播间的主持人是谁？"
        # "《红楼梦》第 88 回里林黛玉对贾宝玉说的那句关于茶的诗是什么?",
    ]
    
    async with asyncio.TaskGroup() as tg:
        tasks_dict: dict[str, list] = {}
        const = "如果你不确定或不知道,请直接说'我不知道',不要编造。"
        for p in prompts:
            task1 = tg.create_task(asyncio.to_thread(chat, input=p, temperature=0.8))
            task2 = tg.create_task(asyncio.to_thread(chat, input=f"{p},{const}", temperature=0.8))
            # task3 = tg.create_task(asyncio.to_thread(chat, input=p, do_sample=False))
            # 收集tasks
            tasks_dict[p] = [task1, task2]
         
    datas: dict[str, list[str]] = {}
    for p, tasks in tasks_dict.items():
        res1: ContentModel = tasks[0].result()
        res2: ContentModel = tasks[1].result()
        # res3: ContentModel = tasks[2].result()
        datas[p] = [res1.content, res2.content]

    df = pd.DataFrame(datas)
    df.insert(loc=0, column="场景", value=["不带约束", "带约束"])
    print(df)

     场景  \
0  不带约束   
1   带约束   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      